In [0]:
# %pip install pymupdf

In [0]:
import logging
import os
import io
import requests
import pandas as pd
import pymupdf
import zipfile
from datetime import datetime

logger = logging.getLogger(__name__)

In [0]:
file_name_index_ticker_map = {
#     'NIFTY 200' : 'Nifty 200',
    'NIFTY 50' : 'Nifty 50',
#     'NIFTY 500' : 'Nifty 500',
#     'NIFTY Auto' : 'Nifty Auto',
    'NIFTY Bank' : 'Nifty Bank',
#     'NIFTY Commodities' : 'Nifty Commodities',
#     'NIFTY Dividend Opportunities 50' : 'Nifty Div Opps 50',
    'NIFTY Financial Services' : 'Nifty Fin Service',
    'NIFTY IT' : 'Nifty IT',
#     'NIFTY India Consumption' : 'Nifty Consumption',
#     'NIFTY Metal' : 'Nifty Metal',
#     'NIFTY Midcap 100' : 'NIFTY MIDCAP 100',
#     'NIFTY PSU Bank' : 'Nifty PSU Bank',
#     'NIFTY Pharma' : 'Nifty Pharma',
#     'NIFTY Quality 30' : 'Nifty Quality 30',
#     'NIFTY Smallcap 100' : 'NIFTY SMLCAP 100',
#     'NIFTY50 Value 20' : 'Nifty50 Value 20',
}

def _validate_date(_date: str) -> None:
    try:
        datetime.strptime(_date, '%Y-%m-%d')
        return None
    except:
        raise TypeError('Invalid Date Format')

def _insert_dataframe_to_db(df: pd.DataFrame, db_name: str, schema_name: str, table_name: str):
    try:
        spark.createDataFrame(df).write.mode('append').format('delta').option('overwriteSchema', 'true').saveAsTable(f'{db_name}.{schema_name}.{table_name}')
        logger.info(f'Inserted {df.shape[0]} rows')
    except Exception as e:
        logger.error(f"Error occurred while inserting symbol changes data into table {table_name}: {e}")
        raise Exception(f"Failed to insert symbol changes data into table {table_name}")

In [0]:
def _parse_index_constituent_makeup_pdf(file_name: str):
    dfs = pd.DataFrame()
    try:
        logger.info(f'Parsing index constituent PDF: {file_name}')
        doc = pymupdf.open(file_name)
        for page in doc:
            for tab in page.find_tables():
                data = tab.extract()
                if not data or len(data) < 2:
                    continue
                df = pd.DataFrame.from_records(data=data[1:], columns=data[0])
                dfs = pd.concat([dfs, df], ignore_index=True)
        if dfs.empty:
            logger.warning(f'No tables found in PDF: {file_name}')
        return dfs
    except Exception as e:
        logger.error(f'Failed to parse PDF {file_name}: {e}')
        raise RuntimeError(f'Error parsing PDF {file_name}')

def _download_index_constituents_makeup(month_year: str):
    url = f"https://www.niftyindices.com/Indices_-_Market_Capitalisation_and_Weightage/indices_data{month_year}.zip"
    session = requests.Session()
    headers = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, \'like Gecko) \'Chrome/80.0.3987.149 Safari/537.36'}

    logger.info(f'Downloading index constituents for {month_year}')
    try:
        response = session.get(url, headers=headers, timeout=10, cookies={})
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error(f'Failed to download index constituents for {month_year}: {e}')
        raise RuntimeError(f'Unable to download index constituents for {month_year}')

    dfs = pd.DataFrame()
    try:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            relevant_files = 0
            for zip_file_obj in z.filelist:
                name_split = ' '.join(zip_file_obj.filename.split('.')[0].split('_')[:-1])
                if name_split not in file_name_index_ticker_map:
                    continue
                relevant_files += 1
                try:
                    z.extract(zip_file_obj.filename)
                except Exception as e:
                    logger.error(f'Failed to extract {zip_file_obj.filename}: {e}')
                    continue
                try:
                    df = _parse_index_constituent_makeup_pdf(zip_file_obj.filename)
                except Exception as e:
                    logger.error(f'Failed to parse PDF {zip_file_obj.filename}: {e}')
                    df = pd.DataFrame()
                finally:
                    try:
                        if os.path.exists(zip_file_obj.filename):
                            os.remove(zip_file_obj.filename)
                    except Exception as e:
                        logger.warning(f'Failed to remove file {zip_file_obj.filename}: {e}')
                if df.empty:
                    logger.warning(f'No data extracted from {zip_file_obj.filename}')
                    continue
                try:
                    df['month_year'] = month_year
                    df['index_name'] = file_name_index_ticker_map.get(name_split, name_split)
                    dfs = pd.concat([dfs, df], ignore_index=True)
                except Exception as e:
                    logger.error(f'Error processing DataFrame for {zip_file_obj.filename}: {e}')
            if relevant_files == 0:
                logger.warning(f'No matching index PDFs found in archive for {month_year}')
    except zipfile.BadZipFile as e:
        logger.error(f'Bad zip archive for {month_year}: {e}')
        raise RuntimeError(f'Invalid zip archive for {month_year}')
    except Exception as e:
        logger.error(f'Error processing archive for {month_year}: {e}')
        raise
    return dfs

def index_constituents_makeup_historical(from_date: str, to_date: str, db_name: str, schema_name: str, table_name: str):
    try:
        _validate_date(from_date)
        _validate_date(to_date)
    except TypeError as e:
        logger.error(f'Invalid date range {from_date} to {to_date}: {e}')
        raise

    if to_date < from_date:
        logger.error(f'From date {from_date} is later than to date {to_date}')
        raise ValueError('To Date should be later than From Date')

    try:
        month_year_list = pd.Series(pd.date_range(from_date, to_date, freq='ME')).dt.strftime('%b%Y').tolist()
        logger.info(f'Retrieving index constituents for {len(month_year_list)} months between {from_date} and {to_date}')
    except Exception as e:
        logger.error(f'Error generating month-year list: {e}')
        raise

    data_df = pd.DataFrame()
    for month_year in month_year_list:
        try:
            dfs = _download_index_constituents_makeup(month_year)
        except Exception as e:
            logger.error(f'Error downloading or parsing data for {month_year}: {e}')
            continue
        if dfs.empty:
            logger.warning(f'No constituent data for {month_year}')
            continue
        try:
            dfs['month_year'] = pd.to_datetime(dfs['month_year'])
            data_df = pd.concat([data_df, dfs], ignore_index=True)
        except Exception as e:
            logger.error(f'Error processing DataFrame for {month_year}: {e}')
            continue

    if data_df.empty:
        logger.warning(f'No historical index constituent data found for range {from_date} to {to_date}')
    try:
        data_df = data_df[['Symbol', 'month_year', 'index_name']].rename(columns = {'Symbol': 'ticker'})
        _insert_dataframe_to_db(data_df[['ticker', 'index_name', 'month_year']], db_name, schema_name, table_name)
    except Exception as e:
        logger.error(f'Error inserting data into database: {e}')
        raise

In [0]:
db_name = 'indian_market'
schema_name = 'nse_india'
table_name = 'index_constituents_makeup'

try:
    from_date = spark.sql(f"SELECT MAX(month_year) as month_year FROM {db_name}.{schema_name}.{table_name}").collect()[0]['month_year'].strftime('%Y-%m-%d')
except Exception as e:
    if e.getCondition() == 'TABLE_OR_VIEW_NOT_FOUND':
        from_date = '2018-01-01'
    else:
        raise e

to_date = datetime.today().strftime('%Y-%m-%d')
logger.info('Filling in from ', from_date, ' to ', to_date)

index_constituents_makeup_historical(from_date=from_date, to_date=to_date, db_name=db_name, schema_name=schema_name, table_name=table_name)